# Experimentos Semana 5

Trabajo de Seminario de Tesis 2: modelos de IA para detección proactiva de fallas físicas en entornos NOC.

Este notebook usa un dataset anonimizado/procesado. No contiene tickets reales, códigos reales de rutas, coordenadas ni nombres de responsables.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score, precision_recall_curve


In [ ]:
DATA_PATH = '../data/processed/incidentes_noc_anon_semana5.csv'
df = pd.read_csv(DATA_PATH)
df.head()


## Definición del target

Target principal: `label_over_time`.

- `1`: incidente Over Time
- `0`: incidente On Time

Para evitar leakage, no se usan como predictores `kpi`, `duration_hours`, `status`, `label_critical` ni `incident_id`.

In [ ]:
target = 'label_over_time'
baseline_features = ['area', 'priority', 'type_of_incident', 'trouble_type', 'incident_type', 'network_id']
var_features = baseline_features + ['year', 'quarter', 'month', 'week_of_year', 'branch_id', 'route_id', 'reason_group']

X = df.drop(columns=[target])
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print(y_train.value_counts(normalize=True))


In [ ]:
def build_pipeline(model, features):
    cat_features = [c for c in features if df[c].dtype == 'object']
    num_features = [c for c in features if c not in cat_features]
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=5), cat_features),
            ('num', StandardScaler(), num_features)
        ],
        remainder='drop'
    )
    
    return Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])

def evaluate(name, pipeline, features):
    pipeline.fit(X_train[features], y_train)
    y_pred = pipeline.predict(X_test[features])
    y_score = pipeline.predict_proba(X_test[features])[:, 1]
    
    return {
        'experimento': name,
        'modelo': pipeline.named_steps['model'].__class__.__name__,
        'accuracy': round(accuracy_score(y_test, y_pred), 4),
        'precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
        'f1': round(f1_score(y_test, y_pred, zero_division=0), 4),
        'average_precision': round(average_precision_score(y_test, y_score), 4),
        'pipeline': pipeline,
        'features': features,
        'y_score': y_score
    }


In [ ]:
experiments = []

experiments.append(evaluate(
    'Baseline',
    build_pipeline(LogisticRegression(max_iter=1000, random_state=42), baseline_features),
    baseline_features
))

experiments.append(evaluate(
    'Var1',
    build_pipeline(RandomForestClassifier(n_estimators=120, max_depth=12, random_state=42, n_jobs=-1), var_features),
    var_features
))

experiments.append(evaluate(
    'Var2',
    build_pipeline(RandomForestClassifier(n_estimators=120, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1), var_features),
    var_features
))

metrics = pd.DataFrame([{k:v for k,v in e.items() if k not in ['pipeline', 'features', 'y_score']} for e in experiments])
metrics


In [ ]:
metrics.to_csv('../results/metricas_semana5.csv', index=False)

plt.figure(figsize=(8, 5))
for e in experiments:
    precision, recall, _ = precision_recall_curve(y_test, e['y_score'])
    plt.plot(recall, precision, label=f"{e['experimento']} AP={e['average_precision']}")

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Semana 5')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/pr_curve_semana5.png', dpi=160)
plt.show()


## Conclusión preliminar

La variante Var2 prioriza la detección de incidentes Over Time mediante balanceo de clases. En contexto NOC, esta estrategia puede ser útil porque reduce el riesgo de no detectar incidentes con posible incumplimiento de KPI.